[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_Pretraining.ipynb)

# 🧠 CerebroSpheroid V5: Endogenous Continued Pretraining (CPT)
### Net2Net Subspace Expansion (v5_small 102M ➔ v5_compact 246M) • Core Freezing & Cyclical Consolidation (Anneal 100 steps every 1,000 steps) • Bioflo.in & Google Drive Sync • Google Colab T4/L4/A100

This notebook executes **Progressive Continued Pretraining (CPT)** to grow your 102M foundation model into a 246M compact flagship:
- **🧬 Net2Net Expansion**: Seeding `v5_compact` (245.6M params: 32 shells, $d=768$, $r=512$, 96 tracts) directly from `v5_small` (102.4M params: 16 shells, $d=640$, $r=256$, 40 tracts), preserving **100% of the 900,000-step TPU foundation priors**.
- **🔒 900-Step Core Freezing**: For steps 0–899 of each 1,000-step window, the inherited core weights remain **FROZEN**. Only the newly added 143M parameters (outer 128 channels, 256 radial channels, shells 16–31, tracts 40–95) train on pretraining tokens.
- **🔓 100-Step Periodic Consolidation Anneal**: For steps 900–999 of each 1,000-step window, the core **UNFREEZES** and the learning rate anneals down along a smooth cosine schedule ($	ext{LR}_{	ext{core}} = 2 	imes 10^{-5}$, $	ext{LR}_{	ext{expansion}} = 2 	imes 10^{-4} 	o 2 	imes 10^{-6}$) to seamlessly co-adapt the new capacity with the core weights.
- **🌐 Automatic Background Cloud Sync**: Auto-mirrors checkpoints (`latest.pt` and `best.pt`) to `bioflo.in` and Google Drive (`/content/drive/MyDrive/cerebro_checkpoints`).

--- 
## 🚀 Step 1: Accelerator Verification (Colab GPU)

In [ ]:
import os, sys, torch

print("=" * 80)
print("⚡ CEREBROSPHEROID V5: ACCELERATOR HARDWARE PROBE")
print("=" * 80)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"✓ GPU Detected       : {gpu_name}")
    print(f"✓ Dedicated VRAM     : {vram_gb:.2f} GB")
    print(f"✓ Native BFloat16    : {'Enabled (Ampere/Ada/Hopper)' if bf16_ok else 'Fallback to FP16 (T4)'}")
    
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
else:
    print("⚠️ Running on CPU! Please select: Runtime -> Change runtime type -> T4 GPU / L4 GPU.")

--- 
## 📦 Step 2: Clone Repository, Install Dependencies & Mount Drive (Optional)

In [ ]:
import os, subprocess

# Clone repository if fresh Colab environment
if not os.path.exists("src/spheroid_v5.py"):
    !git clone https://github.com/AjitKumar-git/cerebro_spheroid_v5.git
    %cd cerebro_spheroid_v5

!pip install -q torch transformers datasets sentencepiece pandas accelerate pyarrow

# Optional: Mount Google Drive for automatic external checkpoint mirror
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted at /content/drive")
except Exception:
    print("ℹ️ Running without Google Drive mount. Checkpoints will be mirrored to bioflo.in and local scratch.")

--- 
## 📥 Step 3: Checkpoint Auto-Discovery & Cloud Resumption

In [ ]:
import os
from src.cloud_sync import check_remote_manifest, pull_checkpoint

os.makedirs("checkpoints", exist_ok=True)
small_ckpt_candidates = [
    "checkpoints/v5_small_latest_step920k.pt",
    "checkpoints/v5_small_best.pt",
    "checkpoints/v5_small_latest_step900k.pt"
]
base_ckpt = next((p for p in small_ckpt_candidates if os.path.exists(p)), None)

if not base_ckpt:
    print("🌐 Checking remote manifest on bioflo.in cloud...")
    manifest = check_remote_manifest(model="v5_small")
    target_file = "checkpoints/v5_small_latest_step920k.pt"
    print(f"📥 Pulling latest v5_small foundation checkpoint into {target_file}...")
    pull_checkpoint(local_dest=target_file, target_type="latest", model="v5_small")
    base_ckpt = target_file

print(f"✓ Foundation model ready: {base_ckpt}")

--- 
## 🧬 Step 4: Execute Continued Pretraining with Cyclical Consolidation

### **Hyperparameters & Dynamic Schedule:**
- **Cycle Length**: Every **1,000 steps**:
  - **Steps 0–899 (900 steps)**: Inherited Core is **FROZEN**. New capacity learns on continuous tokens at $	ext{LR} = 2 	imes 10^{-4}$.
  - **Steps 900–999 (100 steps)**: Core is **UNFROZEN**. Cosine decay anneals learning rate to $2 	imes 10^{-6}$ for weight consolidation.
- **Accumulation**: Micro-batch $2 \times$ Accumulation $8$ = **Effective batch 16** ($8,192$ tokens per optimizer step).
- **Backup**: Checkpoints continuously saved locally, mirrored to `bioflo.in` via `push_checkpoint_async`, and backed up to Google Drive.

In [ ]:
# Launch the Cyclical Growth Pretraining Runner
!python scripts/train_v5_compact_growth_pretrain.py \
    --small-ckpt "checkpoints/v5_small_latest_step920k.pt" \
    --compact-config "configs/v5_compact_250m.json" \
    --output-dir "checkpoints" \
    --total-steps 10000 \
    --cycle-length 1000 \
    --anneal-steps 100 \
    --batch-size 2 \
    --accum-steps 8 \
    --base-lr 2e-4 \
    --min-lr 2e-6 \
    --cloud-sync \
    --device cuda

--- 
## 🔍 Step 5: Test Live Autoregressive Linguistic Generation
Verify the continuous entropy descent and linguistic coherence of the consolidated `v5_compact` model.

In [ ]:
import torch, json
from src.spheroid_v5 import CerebroSpheroidV5, ModelConfigV5
from src.corpus import build_tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = "checkpoints/v5_compact_pretrain_best.pt"
if not os.path.exists(ckpt_path):
    ckpt_path = "checkpoints/v5_compact_pretrain_latest.pt"

print(f"Loading checkpoint: {ckpt_path}")
raw_ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
with open("configs/v5_compact_250m.json") as f:
    cfg = ModelConfigV5(**json.load(f))

model = CerebroSpheroidV5(cfg).to(device)
model.load_state_dict(raw_ckpt["model_state"], strict=False)
model.eval()

tokenizer = build_tokenizer("data/tokenizer")

test_prompts = [
    "In the human brain, cortical layers",
    "The fundamental law of calculus states that",
    "Linear algebra projection matrices are used to",
    "What is 15 + 27? Step-by-step:"
]

print("=" * 80)
print("🧠 CEREBROSPHEROID V5 COMPACT: AUTOREGRESSIVE LINGUISTIC PROBE")
print("=" * 80)

with torch.no_grad():
    for prompt in test_prompts:
        res = model.generate_text(
            prompt,
            tokenizer=tokenizer,
            max_new_tokens=64,
            temperature=0.3,
            device=device,
        )
        print(f"\n👉 Prompt: {prompt}")
        print(f"💡 Output:\n{res.strip()}")
        print("-" * 60)

--- 
## 🌐 Step 6: Verify Remote Bioflo.in & Google Drive Sync

In [ ]:
from src.cloud_sync import check_remote_manifest
import json

print("Remote manifest on bioflo.in for v5_compact:")
try:
    print(json.dumps(check_remote_manifest(model="v5_compact"), indent=2))
except Exception as e:
    print(f"Notice: {e}")

!ls -lh checkpoints/v5_compact_pretrain*.pt
if os.path.exists("/content/drive/MyDrive/cerebro_checkpoints"):
    print("\nGoogle Drive mirrored checkpoints:")
    !ls -lh /content/drive/MyDrive/cerebro_checkpoints/